# 05 · Video Generation

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/sabrinaaquino/base-batches-workshop/blob/main/notebooks/05-video-generation.ipynb)

Video generation is **asynchronous** — you queue a job, then poll for it.

Two endpoints:
- `POST /video/queue` — submits a job, returns a job id
- `POST /video/retrieve` — polls a job, returns status (and the video when ready)

In [ ]:
%pip install --quiet openai requests python-dotenv rich

In [ ]:
import os, sys

# Try Colab secrets first
try:
    from google.colab import userdata  # type: ignore
    api_key = userdata.get("VENICE_API_KEY")
    os.environ["VENICE_API_KEY"] = api_key
except Exception:
    api_key = os.environ.get("VENICE_API_KEY")

if not api_key:
    from getpass import getpass
    api_key = getpass("Paste your Venice API key: ").strip()
    os.environ["VENICE_API_KEY"] = api_key

from openai import OpenAI
client = OpenAI(
    api_key=api_key,
    base_url="https://api.venice.ai/api/v1",
)
print("Connected to Venice ✔")

## 1. Queue a job

In [ ]:
import requests

queued = requests.post(
    "https://api.venice.ai/api/v1/video/queue",
    headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
    json={
        "model": "kling-2.6-pro",
        "prompt": "A vintage venetian gondola gliding through fog at dawn, cinematic, 4 seconds",
        "duration": 4,
    },
    timeout=30,
).json()

print(queued)
job_id = queued["id"]
print(f"\nJob queued: {job_id}")

## 2. Poll until ready

Video generation can take 30–120 seconds depending on model and length. We poll every few seconds and stop on `completed` or `failed`.

In [ ]:
import time

def wait_for_video(job_id: str, every: int = 5, timeout: int = 300):
    start = time.time()
    while time.time() - start < timeout:
        r = requests.post(
            "https://api.venice.ai/api/v1/video/retrieve",
            headers={"Authorization": f"Bearer {api_key}", "Content-Type": "application/json"},
            json={"id": job_id},
            timeout=30,
        ).json()

        status = r.get("status")
        elapsed = int(time.time() - start)
        print(f"[{elapsed:>3}s] status: {status}")

        if status == "completed":
            return r
        if status == "failed":
            raise RuntimeError(f"Job failed: {r}")
        time.sleep(every)
    raise TimeoutError(f"Job {job_id} did not finish in {timeout}s")


result = wait_for_video(job_id)
print("\nResult:")
print(result)

## 3. Download and display

In [ ]:
from IPython.display import Video, display

video_url = result.get("url") or result.get("video_url") or result.get("output", {}).get("url")
print(f"Downloading from: {video_url}")

resp = requests.get(video_url, timeout=120)
with open("gondola.mp4", "wb") as f:
    f.write(resp.content)

print(f"Saved {len(resp.content) / 1024:.0f} KB")
display(Video("gondola.mp4", embed=True))

## What just happened

You submitted an async generation job, polled it to completion, and pulled down a real video file. Same pattern works for **image-to-video** — pass an `image` URL or base64 alongside the prompt.

Cost is roughly proportional to duration; check `x-venice-balance-usd` on responses to track.

**Next:** [06 · Characters](./06-characters.ipynb)